# Brazil adaptation-policy extraction

This notebook extracts the complete native policy structure of the Cities Sectoral Adaptation Plan: every printed objective, target, action, deadline, resource statement, monitoring indicator, and monitoring frequency. Portuguese remains authoritative and every record retains a physical PDF-page locator.

General background prose, analytical risk tables, governance descriptions, and appendix inventories remain source context rather than being converted into policy records. The earlier Climate Plan diagram negative control is retained. Outputs are working samples, not production data.

In [1]:
from pathlib import Path
import json
import re

import pandas as pd
import pdfplumber
from jsonschema import Draft202012Validator

RELEASE_DIR = Path.cwd()
RAW_DIR = RELEASE_DIR / "data" / "raw"
SAMPLE_DIR = RELEASE_DIR / "sample"
SCHEMA_PATH = RELEASE_DIR / "schemas" / "policy_record.schema.json"
CITIES_PATH = RAW_DIR / "br-secadp-cidades-2026.pdf"
NEGATIVE_PATH = RAW_DIR / "br-plano-clima-2026.pdf"
DOCUMENT_ID = "br-secadp-cidades-2026"

assert CITIES_PATH.exists() and NEGATIVE_PATH.exists() and SCHEMA_PATH.exists()
with pdfplumber.open(CITIES_PATH) as pdf:
    profile = {"document_id": DOCUMENT_ID, "pdf_pages": len(pdf.pages), "policy_table_pages": "61–67", "indicator_table_pages": "74–77"}
profile

{'document_id': 'br-secadp-cidades-2026',
 'pdf_pages': 114,
 'policy_table_pages': '61–67',
 'indicator_table_pages': '74–77'}

In [2]:
def normalise(value):
    return re.sub(r"\s+", " ", value or "").strip()


def normalise_pdf_cell(value):
    """Join verified layout hyphenation while preserving genuine compounds found in the source."""
    text = re.sub(r"(?<=\w)-\n(?=\w)", "", value or "")
    text = text.replace("áreaschave", "áreas-chave").replace("póseventos", "pós-eventos")
    return normalise(text)


def record(*, record_id, record_type, native_id, parent_id, source_text, page, document_id=DOCUMENT_ID,
           deadline=None, resources=None, indicators=None, frequency=None,
           notes=None, related_objectives=None, review_note=None, analysis_summary_en=None, section_or_table=None):
    return {
        "schema_version": "0.1.0",
        "record_id": record_id,
        "document_id": document_id,
        "record_type": record_type,
        "source_native_id": native_id,
        "parent_record_id": parent_id,
        "source_text": source_text,
        "analysis_summary_en": analysis_summary_en,
        "deadline": deadline,
        "responsible_institutions": [],
        "resources": resources or [],
        "indicators": indicators or [],
        "monitoring_frequency": frequency,
        "source_notes": notes or [],
        "related_objective_ids": related_objectives or [],
        "pdf_page_start": page,
        "pdf_page_end": page,
        "printed_page": str(page),
        "section_or_table": section_or_table or ("Quadro 5 – Objetivos, metas e ações" if page <= 67 else "Quadro 6 – Metas e indicadores"),
        "evidence_status": "verified",
        "review_status": "proposed",
        "review_note": review_note,
    }


def text_after_id(cell, native_id):
    match = re.search(rf"{re.escape(native_id)}\.?\s*(.+)", normalise(cell))
    assert match, f"Missing native identifier {native_id}"
    return match.group(1).strip()


def split_note(text):
    if "*" not in text:
        return text, []
    body, note = text.split("*", 1)
    return body.rstrip(), [normalise(note)]


print("Helpers loaded; extraction is deterministic and source-bound.")

Helpers loaded; extraction is deterministic and source-bound.


In [3]:
# Extract all three objectives, eight targets, and nineteen actions from Quadro 5.
objective_targets = {"O1": ["M1", "M2", "M3"], "O2": ["M4", "M5", "M6"], "O3": ["M7", "M8"]}
target_parent = {target: objective for objective, targets in objective_targets.items() for target in targets}
records_by_id = {}
source_target_cells = {}

with pdfplumber.open(CITIES_PATH) as pdf:
    for page_number in range(61, 68):
        page = pdf.pages[page_number - 1]
        page_text = page.extract_text() or ""
        tables = page.extract_tables()
        assert len(tables) == 1
        table = tables[0]
        first_cell = normalise(table[0][0])
        objective_in_table = bool(re.search(r"O[1-3]\.\s*", first_cell))
        objective_source = first_cell if objective_in_table else normalise(page_text)
        objective_match = re.search(r"(O[1-3])\.\s*(.+?)(?=\s+(?:Plano Plurianual|Metas relacionadas)|$)", objective_source)
        header_row = 0
        if objective_match:
            native_id, objective_text = objective_match.groups()
            objective_id = f"{DOCUMENT_ID}:{native_id.lower()}"
            related = sorted(set(re.findall(r"\bON\d+\b", page_text)), key=lambda value: int(value[2:]))
            records_by_id[objective_id] = record(
                record_id=objective_id, record_type="objective", native_id=native_id, parent_id=None,
                source_text=objective_text, page=page_number, related_objectives=related,
                review_note="Native identifier used as the boundary after the reversed decorative vertical label.",
            )
            header_row = 1 if objective_in_table else 0
        assert normalise(table[header_row][0]) == "Metas relacionadas"

        current_target = None
        for row in table[header_row + 1:]:
            target_cell, action_cell, resource_cell = row
            if target_cell:
                target_match = re.match(r"(M\d+)\.?\s*(.+?)\s+Prazo:\s*(\d{4})(.*)$", normalise(target_cell))
                assert target_match, normalise(target_cell)
                target_native, target_text, deadline, target_suffix = target_match.groups()
                current_target = target_native
                target_id = f"{DOCUMENT_ID}:{target_native.lower()}"
                source_target_cells.setdefault(target_native, []).append(normalise(target_cell))
                if target_id not in records_by_id:
                    notes = [normalise(target_suffix.lstrip("*"))] if target_suffix.strip() else []
                    records_by_id[target_id] = record(
                        record_id=target_id, record_type="target", native_id=target_native,
                        parent_id=f"{DOCUMENT_ID}:{target_parent[target_native].lower()}", source_text=target_text,
                        page=page_number, deadline=deadline, notes=notes,
                    )
            assert current_target is not None
            action_native_match = re.match(r"(A\d+\.M\d+)\.\s*(.+)", normalise(action_cell))
            assert action_native_match
            action_native, action_text_with_note = action_native_match.groups()
            assert action_native.endswith(f".{current_target}")
            action_text, action_notes = split_note(action_text_with_note)
            action_id = f"{DOCUMENT_ID}:{action_native.lower().replace('.', '-')}"
            assert action_id not in records_by_id
            records_by_id[action_id] = record(
                record_id=action_id, record_type="action", native_id=action_native,
                parent_id=f"{DOCUMENT_ID}:{current_target.lower()}", source_text=action_text,
                page=page_number, resources=[normalise(resource_cell)], notes=action_notes,
                review_note="A blank target cell inherits the last explicit target only within the same visible page table." if target_cell is None else None,
            )

policy_records = list(records_by_id.values())
pd.Series([item["record_type"] for item in policy_records]).value_counts().rename_axis("record_type").to_frame("records")

,records
record_type,
action,19
target,8
objective,3


In [4]:
# Attach the full monitoring table to its eight target records.
indicator_starts = r"(?=Percentual|Número|Quantidade|Valor)"
with pdfplumber.open(CITIES_PATH) as pdf:
    for page_number in range(74, 78):
        tables = pdf.pages[page_number - 1].extract_tables()
        assert len(tables) == 1
        table = tables[0]
        assert normalise(table[0][0]) == "Metas"
        for target_cell, indicator_cell, frequency_cell in table[1:]:
            native_match = re.match(r"(M\d+)\.", normalise(target_cell))
            assert native_match
            target_native = native_match.group(1)
            target_record = records_by_id[f"{DOCUMENT_ID}:{target_native.lower()}"]
            indicator_text = normalise(indicator_cell)
            indicator_notes = []
            note_match = re.search(r"\s+\*Políticas", indicator_text)
            if note_match:
                indicator_notes = [indicator_text[note_match.start():].strip().lstrip("*")]
                indicator_text = indicator_text[:note_match.start()].strip()
            indicators = [part.strip() for part in re.split(rf"(?<=\.)\s+{indicator_starts}", indicator_text) if part.strip()]
            assert indicators
            target_record["indicators"] = indicators
            target_record["monitoring_frequency"] = normalise(frequency_cell)
            target_record["source_notes"].extend(indicator_notes)

target_summary = pd.DataFrame([
    {"target": item["source_native_id"], "indicators_extracted": len(item["indicators"]), "frequency": item["monitoring_frequency"]}
    for item in policy_records if item["record_type"] == "target"
])
target_summary

,target,indicators_extracted,frequency
0,M1,4,Anual
1,M2,2,Anual
2,M3,4,Anual
3,M4,6,Anual
4,M5,5,Anual
5,M6,3,Anual
6,M7,6,Anual
7,M8,1,Anual


In [5]:
# Validate schema, completeness, hierarchy, repeated targets, resources, indicators, and the negative control.
schema = json.loads(SCHEMA_PATH.read_text(encoding="utf-8"))
validator = Draft202012Validator(schema)
schema_errors = [f"{item['record_id']}: {error.message}" for item in policy_records for error in validator.iter_errors(item)]
assert schema_errors == [], schema_errors

expected_actions = {
    "A1.M1", "A2.M1", "A1.M2", "A1.M3", "A1.M4", "A1.M5", "A2.M5", "A3.M5", "A1.M6",
    "A1.M7", "A2.M7", "A3.M7", "A4.M7", "A5.M7", "A6.M7", "A7.M7", "A8.M7", "A9.M7", "A1.M8",
}
assert len(policy_records) == 30
assert {item["source_native_id"] for item in policy_records if item["record_type"] == "objective"} == set(objective_targets)
assert {item["source_native_id"] for item in policy_records if item["record_type"] == "target"} == set(target_parent)
assert {item["source_native_id"] for item in policy_records if item["record_type"] == "action"} == expected_actions
assert len({item["record_id"] for item in policy_records}) == len(policy_records)
assert all(item["parent_record_id"] is None or item["parent_record_id"] in records_by_id for item in policy_records)
assert all(item["deadline"] == "2035" for item in policy_records if item["record_type"] == "target")
assert all(len(item["resources"]) == 1 for item in policy_records if item["record_type"] == "action")
assert all(item["monitoring_frequency"] == "Anual" for item in policy_records if item["record_type"] == "target")
assert sum(len(item["indicators"]) for item in policy_records) == 31
assert len(source_target_cells["M5"]) == 2 and len(source_target_cells["M7"]) == 3

with pdfplumber.open(NEGATIVE_PATH) as pdf:
    negative_tables = pdf.pages[31].extract_tables()
negative_records = []
assert negative_tables and not any("Metas relacionadas" in normalise(" ".join(normalise(c) for row in table for c in row if c)) for table in negative_tables)
assert negative_records == []

validation_summary = {
    "records": len(policy_records),
    "objectives": 3,
    "targets": 8,
    "actions": len(expected_actions),
    "indicator_statements_extracted": sum(len(item["indicators"]) for item in policy_records),
    "indicator_count_stated_in_plan_prose": 23,
    "schema_errors": len(schema_errors),
    "negative_control_records": len(negative_records),
}
validation_summary

{'records': 30,
 'objectives': 3,
 'targets': 8,
 'actions': 19,
 'indicator_statements_extracted': 31,
 'indicator_count_stated_in_plan_prose': 23,
 'schema_errors': 0,
 'negative_control_records': 0}

In [6]:
# Export the working full-document sample; no production dataset is created at this checkpoint.
SAMPLE_DIR.mkdir(exist_ok=True)
output_path = SAMPLE_DIR / "cities_policy_records.jsonl"
output_text = "\n".join(json.dumps(item, ensure_ascii=False, sort_keys=True) for item in policy_records) + "\n"
output_path.write_text(output_text, encoding="utf-8")
round_trip = [json.loads(line) for line in output_path.read_text(encoding="utf-8").splitlines()]
assert round_trip == policy_records
print({"working_output": str(output_path.relative_to(RELEASE_DIR)), "rows": len(round_trip), "production_export": False})

{'working_output': 'sample/cities_policy_records.jsonl', 'rows': 30, 'production_export': False}


In [7]:
# Extract the complete structured policy core of the Portuguese National Adaptation Strategy.
ENA_ID = "br-ena-2025"
ENA_PATH = RAW_DIR / "br-ena-2025.pdf"
ENA_EN_PATH = RAW_DIR / "br-ena-2025-en.pdf"
assert ENA_PATH.exists() and ENA_EN_PATH.exists()
ena_records = []

with pdfplumber.open(ENA_PATH) as pdf:
    # Thirteen numbered adaptation guidelines on physical pages 71–72.
    guideline_chunks = []
    for page_number in (71, 72):
        page_text = normalise(pdf.pages[page_number - 1].extract_text())
        start_phrase = "1. promoção" if page_number == 71 else "7. fortalecimento"
        page_text = page_text[page_text.index(start_phrase):]
        if page_number == 72:
            page_text = page_text.split(" Plano Clima Adaptação", 1)[0]
        matches = list(re.finditer(r"(?<!\d)(\d{1,2})\.\s+(.+?)(?=\s+\d{1,2}\.\s+|$)", page_text))
        for match in matches:
            number, wording = match.groups()
            if int(number) < 13 and ";" in wording:
                wording = wording.split(";", 1)[0] + ";"
            guideline_chunks.append((number, wording.rstrip("; "), page_number))
    assert [number for number, _, _ in guideline_chunks] == [str(number) for number in range(1, 14)]
    for number, wording, page_number in guideline_chunks:
        ena_records.append(record(
            record_id=f"{ENA_ID}:guideline-{number}", record_type="enabling_condition", native_id=number,
            parent_id=None, source_text=wording, page=page_number, document_id=ENA_ID,
            section_or_table="5.1 Diretrizes",
        ))

    # Vision, general objective, and nine numbered national adaptation objectives on physical page 75.
    core_text = normalise(pdf.pages[74].extract_text())
    vision = re.search(r"5\.2\. Visão (.+?) 5\.3\. Objetivo geral", core_text).group(1)
    general_objective = re.search(r"5\.3\. Objetivo geral (.+?) 5\.4\. Objetivos nacionais de adaptação", core_text).group(1)
    objective_block = core_text.split("São eles:", 1)[1].split(" Plano Clima Adaptação", 1)[0]
    objective_matches = list(re.finditer(r"(?<!\d)([1-9])\.\s+(.+?)(?=\s+[1-9]\.\s+|$)", objective_block))
    assert len(objective_matches) == 9
    ena_records.append(record(
        record_id=f"{ENA_ID}:vision", record_type="context", native_id=None, parent_id=None,
        source_text=vision, page=75, document_id=ENA_ID, section_or_table="5.2 Visão",
    ))
    ena_records.append(record(
        record_id=f"{ENA_ID}:general-objective", record_type="objective", native_id=None, parent_id=None,
        source_text=general_objective, page=75, document_id=ENA_ID, section_or_table="5.3 Objetivo geral",
    ))
    for match in objective_matches:
        number, wording = match.groups()
        ena_records.append(record(
            record_id=f"{ENA_ID}:on{number}", record_type="objective", native_id=f"ON{number}",
            parent_id=f"{ENA_ID}:general-objective", source_text=wording.rstrip("; e"), page=75,
            document_id=ENA_ID, section_or_table="5.4 Objetivos nacionais de adaptação",
        ))

    # Footnotes qualify indicators and remain separate from indicator wording.
    footnote_patterns = {
        "14": r"Linha de Base: 13% \(2025\)",
        "15": r"Critério de verificação - resposta afirmativa ao seguinte questionamento: Riscos climáticos foram abordados no Projeto Básico e Executivo do investimento em infraestrutura\?",
        "16": r"Linha de base: 26,39% da Zona Econômica Exclusiva \(ZEE\)",
        "17": r"Parâmetros de conectividade:.+?previsão de definição para início de 2026\.",
        "18": r"Obtido através da compatibilização.+?produção agroecológica\.",
        "19": r"Detalhamento do Indicador:.+?Sistemas Isolados \(Sisol\)\.",
    }
    footnotes = {}
    for page_number in (78, 79, 80):
        page_text = normalise(pdf.pages[page_number - 1].extract_text())
        for marker, pattern in footnote_patterns.items():
            match = re.search(pattern, page_text)
            if match:
                footnotes[marker] = match.group(0)
    assert set(footnotes) == set(footnote_patterns)

    # Twelve national targets and fifteen indicators on physical pages 78–80.
    for page_number in range(78, 81):
        tables = pdf.pages[page_number - 1].extract_tables()
        assert len(tables) == 1
        table = tables[0]
        assert normalise(table[0][0]) == "Metas"
        for number_cell, target_cell, indicator_cell in table[1:]:
            number = normalise(number_cell)
            target_text = normalise(target_cell)
            deadline_match = re.match(r"Até (\d{4}),\s*(.+)", target_text)
            assert deadline_match
            deadline, target_wording = deadline_match.groups()
            indicator_text = normalise(indicator_cell)
            marker_match = re.search(r"(14|15|16|17|18|19)$", indicator_text)
            marker = marker_match.group(1) if marker_match else None
            if marker:
                indicator_text = indicator_text[:marker_match.start()].rstrip(" .") + "."
            indicators = [part.strip() for part in re.split(r"(?=\b\d+\.\s+)", indicator_text) if part.strip()]
            indicators = [re.sub(r"^\d+\.\s+", "", part) for part in indicators]
            ena_records.append(record(
                record_id=f"{ENA_ID}:target-{number}", record_type="target", native_id=number, parent_id=None,
                source_text=target_wording, page=page_number, document_id=ENA_ID, deadline=deadline,
                indicators=indicators, notes=[footnotes[marker]] if marker else [],
                review_note="The Strategy does not provide an explicit target-to-national-objective crosswalk.",
                section_or_table="Quadro 3 – Metas nacionais e indicadores do Plano Clima Adaptação",
            ))

# Align target wording from the official English companion by native target number, never by page number alone.
english_targets = {}
with pdfplumber.open(ENA_EN_PATH) as pdf:
    for page_number in range(77, 80):
        table = pdf.pages[page_number - 1].extract_tables()[0]
        for number_cell, target_cell, _ in table[1:]:
            english_targets[normalise(number_cell)] = normalise(target_cell)
assert set(english_targets) == {str(number) for number in range(1, 13)}
for item in ena_records:
    if item["record_type"] == "target":
        item["analysis_summary_en"] = english_targets[item["source_native_id"]]

pd.Series([item["record_type"] for item in ena_records]).value_counts().rename_axis("record_type").to_frame("records")

,records
record_type,
enabling_condition,13
target,12
objective,10
context,1


In [8]:
# Validate the Strategy separately so its different structure cannot borrow Cities assumptions.
ena_schema_errors = [f"{item['record_id']}: {error.message}" for item in ena_records for error in validator.iter_errors(item)]
assert ena_schema_errors == [], ena_schema_errors
assert len(ena_records) == 36
assert len({item["record_id"] for item in ena_records}) == len(ena_records)
assert {item["source_native_id"] for item in ena_records if item["record_type"] == "objective" and item["source_native_id"]} == {f"ON{number}" for number in range(1, 10)}
assert {item["source_native_id"] for item in ena_records if item["record_type"] == "target"} == {str(number) for number in range(1, 13)}
assert sum(len(item["indicators"]) for item in ena_records) == 15
assert sum(bool(item["source_notes"]) for item in ena_records if item["record_type"] == "target") == 6
assert all(item["analysis_summary_en"] for item in ena_records if item["record_type"] == "target")
assert all(item["parent_record_id"] is None or item["parent_record_id"] in {record["record_id"] for record in ena_records} for item in ena_records)

ena_output_path = SAMPLE_DIR / "national_adaptation_strategy_policy_records.jsonl"
ena_output_text = "\n".join(json.dumps(item, ensure_ascii=False, sort_keys=True) for item in ena_records) + "\n"
ena_output_path.write_text(ena_output_text, encoding="utf-8")
ena_round_trip = [json.loads(line) for line in ena_output_path.read_text(encoding="utf-8").splitlines()]
assert ena_round_trip == ena_records
{
    "records": len(ena_records),
    "guidelines": sum(item["record_type"] == "enabling_condition" for item in ena_records),
    "objectives": sum(item["record_type"] == "objective" for item in ena_records),
    "targets": sum(item["record_type"] == "target" for item in ena_records),
    "indicators": sum(len(item["indicators"]) for item in ena_records),
    "english_target_alignments": sum(bool(item["analysis_summary_en"]) for item in ena_records),
    "schema_errors": len(ena_schema_errors),
    "working_output": str(ena_output_path.relative_to(RELEASE_DIR)),
}

{'records': 36,
 'guidelines': 13,
 'objectives': 10,
 'targets': 12,
 'indicators': 15,
 'english_target_alignments': 12,
 'schema_errors': 0,
 'working_output': 'sample/national_adaptation_strategy_policy_records.jsonl'}

In [9]:
# Extract the complete native policy structure of the Biodiversity thematic plan.
BIO_ID = "br-secadp-biodiversidade-2026"
BIO_PATH = RAW_DIR / "br-secadp-biodiversidade-2026.pdf"
assert BIO_PATH.exists()
bio_records_by_id = {}
bio_target_cells = {}
bio_target_parent = {"M1": "O1", "M2": "O1", "M3": "O1", "M4": "O2", "M5": "O2", "M6": "O3"}
objective_pages = {36: "O1", 41: "O2", 43: "O3"}

with pdfplumber.open(BIO_PATH) as pdf:
    for page_number in range(36, 45):
        page = pdf.pages[page_number - 1]
        page_text = normalise(page.extract_text())
        if page_number in objective_pages:
            objective_native = objective_pages[page_number]
            objective_match = re.search(rf"{objective_native}\.\s*(.+?)(?=\s+Plano Clima|\s+Metas)", page_text)
            assert objective_match
            objective_id = f"{BIO_ID}:{objective_native.lower()}"
            related = sorted(set(re.findall(r"\bON\d+\b", page_text)), key=lambda value: int(value[2:]))
            bio_records_by_id[objective_id] = record(
                record_id=objective_id, record_type="objective", native_id=objective_native, parent_id=None,
                source_text=objective_match.group(1), page=page_number, document_id=BIO_ID,
                related_objectives=related, section_or_table="Quadro 3 – Objetivos, metas e ações",
            )

        tables = page.extract_tables()
        assert len(tables) == 1
        table = tables[0]
        assert [normalise(cell) for cell in table[0]] == ["Metas", "Ações", "Plano Plurianual / Fonte do recurso", "Instituição responsável"]
        current_target = None
        for target_cell, action_cell, resource_cell, institution_cell in table[1:]:
            if target_cell:
                target_value = normalise_pdf_cell(target_cell)
                target_match = re.match(r"(M\d+)\.\s*(.+)", target_value)
                assert target_match
                target_native, target_with_deadline = target_match.groups()
                deadline_match = re.search(r"\baté\s+(2031|2035)\b", target_with_deadline, flags=re.IGNORECASE)
                assert deadline_match
                deadline = deadline_match.group(1)
                target_text = re.sub(r",?\s*até\s+(?:2031|2035),?", "", target_with_deadline, flags=re.IGNORECASE)
                target_text = re.sub(r"\s+,", ",", normalise(target_text)).strip(" ,")
                current_target = target_native
                bio_target_cells.setdefault(target_native, []).append(target_value)
                target_id = f"{BIO_ID}:{target_native.lower()}"
                if target_id not in bio_records_by_id:
                    bio_records_by_id[target_id] = record(
                        record_id=target_id, record_type="target", native_id=target_native,
                        parent_id=f"{BIO_ID}:{bio_target_parent[target_native].lower()}", source_text=target_text,
                        page=page_number, document_id=BIO_ID, deadline=deadline,
                        section_or_table="Quadro 3 – Objetivos, metas e ações",
                    )
            assert current_target is not None
            action_value = normalise_pdf_cell(action_cell)
            action_match = re.match(r"A(\d+)\.\s*M(\d+)\.\s*(.+)", action_value)
            assert action_match, action_value
            action_number, target_number, action_text = action_match.groups()
            action_native = f"A{action_number}.M{target_number}"
            assert action_native.endswith(f".{current_target}")
            action_id = f"{BIO_ID}:{action_native.lower().replace('.', '-')}"
            assert action_id not in bio_records_by_id
            action_record = record(
                record_id=action_id, record_type="action", native_id=action_native,
                parent_id=f"{BIO_ID}:{current_target.lower()}", source_text=action_text,
                page=page_number, document_id=BIO_ID, resources=[normalise_pdf_cell(resource_cell)],
                section_or_table="Quadro 3 – Objetivos, metas e ações",
                review_note="A blank target cell inherits the last explicit target only within the same visible page table." if target_cell is None else None,
            )
            action_record["responsible_institutions"] = [normalise_pdf_cell(institution_cell)]
            bio_records_by_id[action_id] = action_record

bio_records = list(bio_records_by_id.values())
pd.Series([item["record_type"] for item in bio_records]).value_counts().rename_axis("record_type").to_frame("records")

,records
record_type,
action,31
target,6
objective,3


In [10]:
# Attach monitoring indicators and validate Biodiversity independently.
with pdfplumber.open(BIO_PATH) as pdf:
    for page_number in (50, 51):
        table = pdf.pages[page_number - 1].extract_tables()[0]
        assert [normalise(cell) for cell in table[0]] == ["Metas", "Indicadores das metas", "Periodicidade de coleta do indicador"]
        for target_cell, indicator_cell, frequency_cell in table[1:]:
            target_native = re.match(r"(M\d+)\.", normalise_pdf_cell(target_cell)).group(1)
            target_record = bio_records_by_id[f"{BIO_ID}:{target_native.lower()}"]
            indicator_text = normalise_pdf_cell(indicator_cell)
            indicators = [part.strip() for part in re.split(r"(?<=\.)\s+(?=Número|Proporção|%|Área)", indicator_text) if part.strip()]
            target_record["indicators"] = indicators
            target_record["monitoring_frequency"] = normalise_pdf_cell(frequency_cell)

expected_action_counts = {"M1": 11, "M2": 2, "M3": 4, "M4": 4, "M5": 3, "M6": 7}
expected_actions = {f"A{number}.{target}" for target, count in expected_action_counts.items() for number in range(1, count + 1)}
bio_schema_errors = [f"{item['record_id']}: {error.message}" for item in bio_records for error in validator.iter_errors(item)]
assert bio_schema_errors == [], bio_schema_errors
assert len(bio_records) == 40
assert {item["source_native_id"] for item in bio_records if item["record_type"] == "objective"} == {"O1", "O2", "O3"}
assert {item["source_native_id"] for item in bio_records if item["record_type"] == "target"} == set(bio_target_parent)
assert {item["source_native_id"] for item in bio_records if item["record_type"] == "action"} == expected_actions
assert len({item["record_id"] for item in bio_records}) == len(bio_records)
assert all(item["parent_record_id"] is None or item["parent_record_id"] in bio_records_by_id for item in bio_records)
assert all(len(item["resources"]) == 1 and len(item["responsible_institutions"]) == 1 for item in bio_records if item["record_type"] == "action")
assert sum(len(item["indicators"]) for item in bio_records) == 10
assert {item["monitoring_frequency"] for item in bio_records if item["record_type"] == "target"} == {"2 anos", "4 anos", "5 anos"}
assert len(bio_target_cells["M1"]) == 3 and len(bio_target_cells["M3"]) == 2 and len(bio_target_cells["M6"]) == 2
assert not any(re.search(r"\w-\s+\w", item["source_text"]) for item in bio_records)

bio_output_path = SAMPLE_DIR / "biodiversity_policy_records.jsonl"
bio_output_text = "\n".join(json.dumps(item, ensure_ascii=False, sort_keys=True) for item in bio_records) + "\n"
bio_output_path.write_text(bio_output_text, encoding="utf-8")
bio_round_trip = [json.loads(line) for line in bio_output_path.read_text(encoding="utf-8").splitlines()]
assert bio_round_trip == bio_records
{
    "records": len(bio_records),
    "objectives": 3,
    "targets": 6,
    "actions": len(expected_actions),
    "indicators": sum(len(item["indicators"]) for item in bio_records),
    "responsible_action_records": sum(bool(item["responsible_institutions"]) for item in bio_records),
    "schema_errors": len(bio_schema_errors),
    "working_output": str(bio_output_path.relative_to(RELEASE_DIR)),
}

{'records': 40,
 'objectives': 3,
 'targets': 6,
 'actions': 31,
 'indicators': 10,
 'responsible_action_records': 31,
 'schema_errors': 0,
 'working_output': 'sample/biodiversity_policy_records.jsonl'}

In [11]:
# Generalized extraction for the five remaining sectoral and thematic plans.
def native_action(cell):
    match = re.match(r"A\s*(\d+)\s*\.?\s*M\s*(\d+)\s*(?:[–-]|\.|:)?\s*(.+)", normalise_pdf_cell(cell), flags=re.IGNORECASE)
    assert match, normalise_pdf_cell(cell)
    action_number, target_number, wording = match.groups()
    return f"A{int(action_number)}.M{int(target_number)}", wording


def target_value(cell):
    match = re.match(r"M\s*(\d+)\s*\.?(.*)", normalise_pdf_cell(cell), flags=re.IGNORECASE)
    assert match, normalise_pdf_cell(cell)
    number, wording = match.groups()
    native_id = f"M{int(number)}"
    deadline_match = re.search(r"\b(?:até|ano de)\s+(20\d{2})\b", wording, flags=re.IGNORECASE)
    return native_id, wording.strip(" ."), deadline_match.group(1) if deadline_match else None


def accepted_policy_table(table):
    header = " | ".join(normalise(cell) for cell in table[0] if cell)
    return "Ações" in header and "Meta" in header and "Fonte do recurso" in header


def accepted_indicator_table(table):
    header = " | ".join(normalise(cell) for cell in table[0] if cell)
    return "Meta" in header and "Indicador" in header and "Periodicidade" in header


remaining_configs = {
    "br-secadp-energia-2026": {
        "file": "br-secadp-energia-2026.pdf", "policy_pages": range(51, 67), "indicator_pages": range(73, 76),
        "parents": {"M1": ["O3"], "M2": ["O3"], "M3": ["O1", "O3"], "M4": ["O1", "O3"], "M5": ["O1"], "M6": ["O1", "O2", "O3"], "M7": ["O1", "O3"], "M8": ["O1", "O3"], "M9": ["O1", "O3"], "M10": ["O2"], "M11": ["O2"], "M12": ["O1", "O3"], "M13": ["O3"], "M14": ["O1", "O3"], "M15": ["O1", "O3"], "M16": ["O1", "O3"]},
        "output": "energy_policy_records.jsonl",
    },
    "br-secadp-recursos-hidricos-2026": {
        "file": "br-secadp-recursos-hidricos-2026.pdf", "policy_pages": range(54, 66), "indicator_pages": [71],
        "parents": {**{f"M{number}": ["O1"] for number in (1, 2)}, "M3": ["O2"], **{f"M{number}": ["O3"] for number in (4, 5, 6)}},
        "output": "water_resources_policy_records.jsonl",
    },
    "br-secadp-riscos-desastres-2026": {
        "file": "br-secadp-riscos-desastres-2026.pdf", "policy_pages": range(49, 71), "indicator_pages": range(75, 77),
        "parents": {**{f"M{number}": ["O1"] for number in range(1, 5)}, **{f"M{number}": ["O2"] for number in range(5, 7)}, **{f"M{number}": ["O3"] for number in range(7, 11)}},
        "output": "disaster_risk_policy_records.jsonl",
    },
    "br-secadp-saude-2026": {
        "file": "br-secadp-saude-2026.pdf", "policy_pages": range(51, 79), "indicator_pages": range(82, 88),
        "parents": {**{f"M{number}": ["O1"] for number in range(1, 7)}, **{f"M{number}": ["O2"] for number in range(7, 15)}, **{f"M{number}": ["O3"] for number in range(15, 20)}, **{f"M{number}": ["O4"] for number in range(20, 28)}},
        "output": "health_policy_records.jsonl",
    },
    "br-secadp-seguranca-alimentar-2026": {
        "file": "br-secadp-seguranca-alimentar-2026.pdf", "policy_pages": range(38, 56), "indicator_pages": range(60, 68),
        "parents": {**{f"M{number}": ["O1"] for number in range(1, 7)}, **{f"M{number}": ["O2"] for number in range(7, 19)}, **{f"M{number}": ["O3"] for number in range(19, 22)}, **{f"M{number}": ["O4"] for number in range(22, 25)}, **{f"M{number}": ["O5"] for number in range(25, 35)}},
        "output": "food_nutrition_security_policy_records.jsonl",
    },
}

remaining_sector_results = {}
remaining_sector_records = {}
for document_id, config in remaining_configs.items():
    path = RAW_DIR / config["file"]
    records_by_id = {}
    expected_actions = set()
    expected_targets = set()
    objective_wording = {}
    with pdfplumber.open(path) as pdf:
        # Recover source-defined sector objectives wherever their full heading is printed.
        for page_number in config["policy_pages"]:
            page_text = normalise(pdf.pages[page_number - 1].extract_text())
            for match in re.finditer(r"\b(O\d+)\.\s*(.+?)(?=\s+O\d+\.|\s+Plano Plurianual|\s+Metas(?: relacionadas)?\s+Ações|\s+Plano Clima)", page_text):
                objective_wording.setdefault(match.group(1), (match.group(2), page_number, sorted(set(re.findall(r"\bON\d+\b", page_text)))))
        for objective_native, (wording, page_number, related) in objective_wording.items():
            objective_id = f"{document_id}:{objective_native.lower()}"
            records_by_id[objective_id] = record(
                record_id=objective_id, record_type="objective", native_id=objective_native, parent_id=None,
                source_text=wording, page=page_number, document_id=document_id, related_objectives=related,
                section_or_table="Objetivos setoriais / Metas / Ações",
            )

        for page_number in config["policy_pages"]:
            for table in pdf.pages[page_number - 1].extract_tables():
                if not accepted_policy_table(table):
                    continue
                for row in table[1:]:
                    if len(row) < 3 or not row[1]:
                        continue
                    if row[0]:
                        target_native, target_text, deadline = target_value(row[0])
                        expected_targets.add(target_native)
                        target_id = f"{document_id}:{target_native.lower()}"
                        if target_id not in records_by_id:
                            related_sector_objectives = config["parents"].get(target_native, [])
                            parent_id = f"{document_id}:{related_sector_objectives[0].lower()}" if len(related_sector_objectives) == 1 else None
                            records_by_id[target_id] = record(
                                record_id=target_id, record_type="target", native_id=target_native, parent_id=parent_id,
                                source_text=target_text, page=page_number, document_id=document_id, deadline=deadline,
                                related_objectives=related_sector_objectives if len(related_sector_objectives) > 1 else [],
                                review_note="The source prints several sector objectives for this target; no single parent is assigned." if len(related_sector_objectives) > 1 else None,
                                section_or_table="Metas / Ações / Fonte do recurso",
                            )
                    action_native, action_text = native_action(row[1])
                    expected_actions.add(action_native)
                    target_native = action_native.split(".")[1]
                    action_id = f"{document_id}:{action_native.lower().replace('.', '-')}"
                    if action_id in records_by_id:
                        existing = records_by_id[action_id]
                        existing["source_text"] = normalise(f"{existing['source_text']} {action_text}")
                        resource_fragment = normalise_pdf_cell(row[2])
                        if resource_fragment not in existing["resources"]:
                            existing["resources"].append(resource_fragment)
                        existing["pdf_page_end"] = page_number
                        existing["review_note"] = "The source action continues across consecutive table pages; fragments are joined in page order."
                        continue
                    action_record = record(
                        record_id=action_id, record_type="action", native_id=action_native,
                        parent_id=f"{document_id}:{target_native.lower()}", source_text=action_text,
                        page=page_number, document_id=document_id, resources=[normalise_pdf_cell(row[2])],
                        section_or_table="Metas / Ações / Fonte do recurso",
                    )
                    if len(row) > 3 and row[3]:
                        action_record["responsible_institutions"] = [normalise_pdf_cell(row[3])]
                    records_by_id[action_id] = action_record

        # Attach each target's monitoring cell and frequency without forcing an uncertain sub-indicator count.
        for page_number in config["indicator_pages"]:
            for table in pdf.pages[page_number - 1].extract_tables():
                if not accepted_indicator_table(table):
                    continue
                for row in table[1:]:
                    if len(row) < 3 or not row[0]:
                        continue
                    target_native, _, _ = target_value(row[0])
                    target_id = f"{document_id}:{target_native.lower()}"
                    assert target_id in records_by_id
                    records_by_id[target_id]["indicators"] = [normalise_pdf_cell(row[1])] if row[1] else []
                    records_by_id[target_id]["monitoring_frequency"] = normalise_pdf_cell(row[2]) if row[2] else None

    records = list(records_by_id.values())
    parsed_actions = {item["source_native_id"] for item in records if item["record_type"] == "action"}
    parsed_targets = {item["source_native_id"] for item in records if item["record_type"] == "target"}
    errors = [f"{item['record_id']}: {error.message}" for item in records for error in validator.iter_errors(item)]
    assert not errors, errors
    assert parsed_actions == expected_actions
    assert parsed_targets == expected_targets == set(config["parents"])
    assert all(item["parent_record_id"] is None or item["parent_record_id"] in records_by_id for item in records)
    assert all(item["indicators"] for item in records if item["record_type"] == "target")
    output_path = SAMPLE_DIR / config["output"]
    output_path.write_text("\n".join(json.dumps(item, ensure_ascii=False, sort_keys=True) for item in records) + "\n", encoding="utf-8")
    assert len(output_path.read_text(encoding="utf-8").splitlines()) == len(records)
    remaining_sector_records[document_id] = records
    remaining_sector_results[document_id] = {
        "objectives": sum(item["record_type"] == "objective" for item in records),
        "targets": len(parsed_targets), "actions": len(parsed_actions), "records": len(records),
        "target_monitoring_cells": sum(bool(item["indicators"]) for item in records), "schema_errors": len(errors),
    }

pd.DataFrame.from_dict(remaining_sector_results, orient="index")

,objectives,targets,actions,records,target_monitoring_cells,schema_errors
br-secadp-energia-2026,3,16,38,57,16,0
br-secadp-recursos-hidricos-2026,3,6,40,49,6,0
br-secadp-riscos-desastres-2026,3,10,89,102,10,0
br-secadp-saude-2026,4,27,93,124,27,0
br-secadp-seguranca-alimentar-2026,5,34,60,99,34,0


In [12]:
# Extract only document-specific adaptation commitments from the NDC and the executive summary.
# Reproduced ENA guidelines/objectives are not emitted again from the summary; its national target table is retained as a restatement.
NDC_ID = "br-ndc-2024"
NDC_PATH = RAW_DIR / "br-ndc-2024.pdf"
SUMMARY_ID = "br-plano-clima-2026"
SUMMARY_PATH = RAW_DIR / "br-plano-clima-2026.pdf"

ndc_records = []
with pdfplumber.open(NDC_PATH) as pdf:
    ndc_text = normalise(pdf.pages[34].extract_text())
    planning = re.search(r"In terms of adaptation.+?temperature goals\.", ndc_text).group(0)
    mainstreaming = re.search(r"The effective implementation of the National Adaptation Strategy.+?civil society organizations\.", ndc_text).group(0)
    new_actions = re.search(r"This implies reviewing, reorienting and resizing policies, programs and initiatives\..+?sectoral and thematic plans\.", ndc_text).group(0)
    means = re.search(r"For the implementation of mitigation and adaptation actions,.+?Response Measures \(paragraphs 136-152 of decision 1/CMA\.5\)\.", ndc_text).group(0)
for native_id, record_type, wording, section in [
    ("adaptation-1", "context", planning, "Adaptation"),
    ("adaptation-2", "action", mainstreaming, "Adaptation"),
    ("adaptation-3", "action", new_actions, "Adaptation"),
    ("means-1", "enabling_condition", means, "Need for means of implementation"),
]:
    ndc_records.append(record(
        record_id=f"{NDC_ID}:{native_id}", record_type=record_type, native_id=None, parent_id=None,
        source_text=wording, page=35, document_id=NDC_ID, section_or_table=section,
        review_note="Narrative commitment: the NDC does not assign an O/M/A identifier.",
    ))

summary_records = []
with pdfplumber.open(SUMMARY_PATH) as pdf:
    for page_number in (30, 31):
        tables = pdf.pages[page_number - 1].extract_tables()
        assert len(tables) == 1
        for target_cell, indicator_cell in tables[0]:
            target_match = re.match(r"(\d+)\.\s*Até\s+(\d{4}),\s*(.+)", normalise_pdf_cell(target_cell))
            assert target_match
            number, deadline, wording = target_match.groups()
            summary_records.append(record(
                record_id=f"{SUMMARY_ID}:target-{number}", record_type="target", native_id=number, parent_id=None,
                source_text=wording, page=page_number, document_id=SUMMARY_ID, deadline=deadline,
                indicators=[normalise_pdf_cell(indicator_cell)],
                section_or_table="Quadro 1 – Metas Nacionais de Adaptação e respectivos indicadores",
                review_note="Executive-summary restatement of a national target; deduplicate by instrument and native target number for cross-document analysis.",
            ))

ndc_errors = [error.message for item in ndc_records for error in validator.iter_errors(item)]
summary_errors = [error.message for item in summary_records for error in validator.iter_errors(item)]
assert not ndc_errors and not summary_errors
assert len(ndc_records) == 4
assert len(summary_records) == 12 and {item["source_native_id"] for item in summary_records} == {str(number) for number in range(1, 13)}
assert sum(len(item["indicators"]) for item in summary_records) == 12

for output_name, records in [("ndc_adaptation_policy_records.jsonl", ndc_records), ("climate_plan_summary_adaptation_policy_records.jsonl", summary_records)]:
    output_path = SAMPLE_DIR / output_name
    output_path.write_text("\n".join(json.dumps(item, ensure_ascii=False, sort_keys=True) for item in records) + "\n", encoding="utf-8")
    assert len(output_path.read_text(encoding="utf-8").splitlines()) == len(records)

{
    "ndc_document_specific_adaptation_records": len(ndc_records),
    "executive_summary_national_target_restatements": len(summary_records),
    "schema_errors": len(ndc_errors) + len(summary_errors),
}

{'ndc_document_specific_adaptation_records': 4,
 'executive_summary_national_target_restatements': 12,
 'schema_errors': 0}

## Required English review phase

Run this phase only after the Portuguese extraction and source checks pass. Translations are stored separately and never replace Portuguese evidence. A record is not ready for English-language review until every non-empty Portuguese descriptive field has an English companion.

The translation script is resumable and validates list lengths and numeric content. It uses official English wording for the twelve Strategy targets when the review CSV is built.

In [ ]:
# Requires OPENAI_API_KEY in the notebook environment.
import subprocess
import sys

translation_script = RELEASE_DIR / "scripts" / "translate_policy_records.py"
review_csv_script = RELEASE_DIR / "scripts" / "build_review_csv.py"
subprocess.run([sys.executable, str(translation_script)], check=True)
subprocess.run([sys.executable, str(translation_script), "--validate-only"], check=True)
subprocess.run([sys.executable, str(review_csv_script)], check=True)


## Findings after the complete ten-document corpus

- Complete native policy structure: 3 objectives, 8 targets, and 19 actions, for 30 records.
- Every target has a 2035 deadline and annual monitoring frequency. Every action retains its same-row resource statement.
- Repeated target cells on continuation pages are reconciled rather than duplicated: `M5` appears on two policy-table pages and `M7` on three.
- The monitoring cells contain 31 separately punctuated indicator statements, although the plan's introductory prose says that 23 indicators were developed. The extraction preserves the 31 visible statements and flags the source-side discrepancy rather than silently forcing the stated count.
- The explanatory notes for `A3.M7`, `M8`, and the `M5` indicators are retained separately from the core policy wording.
- The Climate Plan framework diagram still emits zero policy records.
- The National Adaptation Strategy adds 36 records: 13 adaptation guidelines, one vision statement, one general objective, nine national adaptation objectives, and 12 national targets with 15 indicators.
- The Strategy does not explicitly map its 12 targets to its nine national objectives. Those relationships remain unassigned rather than inferred from topic similarity.
- All 12 Strategy targets are aligned by native target number to their official English companion wording. Portuguese remains the evidence source; English is an analysis aid.
- Six Strategy targets retain source footnotes containing baselines, verification criteria, parameters, data derivation, or calculation detail.
- The Biodiversity plan adds 40 records: three objectives, six targets, and 31 actions, with ten linked indicators.
- Every Biodiversity action retains both its same-row resource statement and responsible institution wording; monitoring frequencies vary between two, four, and five years.
- Narrow Biodiversity table columns insert visual line-break hyphens inside ordinary words. The cleanup joins verified layout breaks while explicitly restoring the genuine source compounds `áreas-chave` and `pós-eventos`.
- Repeated Biodiversity targets are reconciled across continuation pages: `M1` spans three pages, while `M3` and `M6` span two pages each.
- The five remaining structured plans add 431 records: Energy 57, Water Resources 49, Disaster Risk 102, Health 124, and Food and Nutrition Security 99.
- Across all seven included sectoral and thematic plans, the extraction contains 107 targets and 370 actions. These are subset counts, not the official full-framework totals of 312 targets and 810 actions across sixteen plans.
- Energy action `A2.M10` is a genuine two-page continuation. Its wording and resource fragments are joined in page order and its evidence range spans both pages.
- For the five newly generalized sector plans, each target's complete monitoring cell is retained as one source unit. No uncertain sub-indicator count is inferred from punctuation.
- The NDC contributes four document-specific narrative adaptation records because it has no native objective/target/action identifiers.
- The Climate Plan executive summary contributes twelve national-target restatements. Its repeated ENA guidelines and objectives are not emitted again, and the target records carry a cross-document deduplication warning.
- The ten working outputs contain 553 document records in total. This is not a deduplicated policy count because the Strategy and executive summary both publish the twelve national targets.

All ten canonical documents in scope now have validated adaptation extraction outputs. The separate English review phase produces bilingual, review-facing records without replacing Portuguese evidence. After English quality review, the next phase is adaptation-action matching.